# Data Preprocessing & Merging Pipeline (V2 - Natural Merge & Robust Imputation)

Notebook ini mengintegrasikan data pemeriksaan laboratorium dan data registrasi poliklinik/dokter secara **100% natural berbasis `NO_REG`**:
1. **Sinkronisasi Data Dokter & Lab**: Seluruh 20 pasien `SPES. PARU PARU` terintegrasi secara alami melalui `NO_REG` registrasi rumah sakit dan pemeriksaan laboratorium standar.
2. **Pembersihan Kelas Kontrol Tanpa Default 0**: Pasien pada Kelas 0 (Sehat/Kontrol) yang tidak memiliki nomor rekam medis poliklinik atau berumur $\le 0$ dihapus. Karena populasi kontrol sangat melimpah (>3.500 data), eliminasi baris cacat ini menghasilkan **2.812 pasien kontrol dengan demografi riil lengkap (usia 1–94 tahun)** tanpa manipulasi default 0.
3. **Robust Imputation (Anti-Zero Fallback)**: Menggunakan KNN Cosine Similarity per kelas dengan **Global Population Median Fallback** untuk menjamin tidak ada nilai `0` artifisial pada parameter klinis.

## 1. Import Library & Konfigurasi Lingkungan

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Set opsi display pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load Data Laboratorium Terpilih & Seluruh Arsip Dokter

In [17]:
# 1. Load Data Lab yang telah dipivot dan dibersihkan
LAB_PATH = 'Cleaning/df_lab_selected.csv' if os.path.exists('Cleaning/df_lab_selected.csv') else 'df_lab_selected.csv'
df_lab = pd.read_csv(LAB_PATH)
df_lab['NO_REG'] = df_lab['NO_REG'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
print(f'Data Lab loaded: {df_lab.shape[0]} baris, {df_lab.shape[1]} kolom')

# 2. Load seluruh file Dokter untuk mendapatkan profil demografi lengkap
DOKTER_DIR = 'Dokter' if os.path.exists('Dokter') else '.'
doc_files = ['dokter_feb.csv', 'DOKTER_MARET-APRIL06.csv', 'DOKTER_MEI-JUNI06.csv']
docs = []
for fname in doc_files:
    fpath = os.path.join(DOKTER_DIR, fname)
    if os.path.exists(fpath):
        d = pd.read_csv(fpath, encoding='latin1')
        d.columns = [c.replace('\ufeff', '').replace('\xef\xbb\xbf', '').strip() for c in d.columns]
        docs.append(d[['NO_REG', 'JENIS_KELAMIN', 'UMUR_TAHUN', 'NM_UNIT']])

df_doc_all = pd.concat(docs, ignore_index=True).dropna(subset=['NO_REG'])
df_doc_all['NO_REG'] = df_doc_all['NO_REG'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
print(f'Data Registrasi Dokter gabungan: {len(df_doc_all)} total record, {df_doc_all["NO_REG"].nunique()} pasien unik')

Data Lab loaded: 4170 baris, 114 kolom
Data Registrasi Dokter gabungan: 94918 total record, 40821 pasien unik


## 3. Deduplikasi & Prioritisasi Poli Spesialis Target
Jika seorang pasien memiliki lebih dari satu kunjungan poli, utamakan registrasi di poli spesialis target (Jantung, Paru, Penyakit Dalam).

In [18]:
TARGET_UNITS = ['JANTUNG DAN PEMBULUH DARAH', 'SPES. PENYAKIT DALAM', 'SPES. PARU PARU']
df_doc_all['is_target'] = df_doc_all['NM_UNIT'].isin(TARGET_UNITS)

# Urutkan berdasarkan prioritas poli target lalu usia
df_doc_dedup = df_doc_all.sort_values(by=['is_target', 'UMUR_TAHUN'], ascending=[False, False])
df_doc_dedup = df_doc_dedup.drop_duplicates(subset=['NO_REG'], keep='first')
print(f'Data profil demografi dokter unik siap di-merge: {len(df_doc_dedup)} pasien')

Data profil demografi dokter unik siap di-merge: 40821 pasien


## 4. Penggabungan Natural Data Lab & Demografi Dokter (Merge on `NO_REG`)

In [19]:
# Left merge pasien lab dengan profil dokter berbasis NO_REG secara natural
df_merged = pd.merge(df_lab, df_doc_dedup, on='NO_REG', how='left')

# Mapping label target spesialisasi
mapping_nm_unit = {
    'JANTUNG DAN PEMBULUH DARAH': 1,
    'SPES. PENYAKIT DALAM': 2,
    'SPES. PARU PARU': 3
}
df_merged['NM_UNIT'] = df_merged['NM_UNIT'].map(mapping_nm_unit).fillna(0).astype(int)

print(f'Hasil merge alami: {df_merged.shape[0]} baris')
print('Distribusi target hasil merge:')
print(df_merged['NM_UNIT'].value_counts().sort_index())

Hasil merge alami: 4170 baris
Distribusi target hasil merge:
NM_UNIT
0    3537
1      52
2     561
3      20
Name: count, dtype: int64


## 5. Pembersihan Data Kelas Kontrol (Hapus Tanpa Default 0 Artifisial)
Karena data pasien sehat/kontrol sangat melimpah (>3.500 baris), kita tidak perlu memaksakan default 0 pada pasien yang tidak memiliki catatan dokter atau bayi (usia $\le 0$). Kita **hapus baris kelas 0 yang tidak lengkap**, sementara seluruh pasien spesialis target (kelas 1, 2, 3) dipertahankan 100%.

In [20]:
# Hapus baris di Kelas 0 yang tidak memiliki umur atau umur <= 0
valid_mask = (df_merged['NM_UNIT'] > 0) | (df_merged['UMUR_TAHUN'].notna() & (df_merged['UMUR_TAHUN'] > 0))
df_filtered = df_merged[valid_mask].copy()

# Samakan penamaan glukosa darah ke fbs & rbs
rename_dict = {}
if 'glukosa darah puasa' in df_filtered.columns:
    rename_dict['glukosa darah puasa'] = 'fbs'
if 'glukosa darah sewaktu' in df_filtered.columns:
    rename_dict['glukosa darah sewaktu'] = 'rbs'
df_filtered = df_filtered.rename(columns=rename_dict)

# Seleksi 13 fitur kanonikal + 1 target
CANONICAL_FEATURES = [
    'JENIS_KELAMIN',
    'UMUR_TAHUN',
    'cholesterol total',
    'creatinin',
    'fbs',
    'rbs',
    'hgb',
    'lymfosit%',
    'mch',
    'mchc',
    'mcv',
    'ureum',
    'wbc',
    'NM_UNIT'
]
df_selected = df_filtered[CANONICAL_FEATURES].copy()

# Encode gender: L = 0, P = 1
df_selected['JENIS_KELAMIN'] = df_selected['JENIS_KELAMIN'].map({'L': 0, 'P': 1}).astype(int)

print(f'Dimensi dataset setelah pembersihan kontrol cacat: {df_selected.shape[0]} baris')
print('Distribusi target final:')
print(df_selected['NM_UNIT'].value_counts().sort_index())

# Simpan dataset mentah WithLab sebelum imputasi
df_selected.to_csv('merge_DocLab_WithLab.csv', index=False)
print('Berhasil menyimpan: merge_DocLab_WithLab.csv')

Dimensi dataset setelah pembersihan kontrol cacat: 3445 baris
Distribusi target final:
NM_UNIT
0    2812
1      52
2     561
3      20
Name: count, dtype: int64
Berhasil menyimpan: merge_DocLab_WithLab.csv


## 6. Imputasi Missing Value: Cosine Similarity KNN dengan Global Median Fallback

In [21]:
feature_cols = [c for c in df_selected.columns if c != 'NM_UNIT']
global_medians = df_selected[feature_cols].median(numeric_only=True)
print('=== Global Population Medians (Fallback) ===')
print(global_medians.round(2))

def impute_missing_by_class(group, features, global_fallback, k=5):
    X = group[features].copy()
    col_medians = X.median(numeric_only=True)
    # Fallback bertingkat: Median Kelas -> Median Global
    fallback = col_medians.fillna(global_fallback)
    
    if len(X) <= 1:
        return X.fillna(fallback)
    
    X_reference = X.fillna(fallback)
    similarity = cosine_similarity(X_reference.values)
    np.fill_diagonal(similarity, -np.inf)
    
    imputed = X.copy()
    for col in features:
        missing_mask = imputed[col].isna()
        if not missing_mask.any():
            continue
            
        known_indices = np.where(~imputed[col].isna())[0]
        if len(known_indices) == 0:
            imputed[col] = imputed[col].fillna(fallback[col])
            continue
            
        for idx in np.where(missing_mask)[0]:
            sim_scores = similarity[idx, known_indices]
            top_k_indices = known_indices[np.argsort(sim_scores)[-min(k, len(sim_scores)):]]
            imputed.iloc[idx, imputed.columns.get_loc(col)] = group.iloc[top_k_indices][col].mean()
            
    return imputed.fillna(fallback)

# Jalankan imputasi per kelompok kelas target
processed_groups = []
for unit, group in df_selected.groupby('NM_UNIT', group_keys=False):
    imputed_subset = impute_missing_by_class(group, feature_cols, global_medians, k=5)
    imputed_subset['NM_UNIT'] = unit
    processed_groups.append(imputed_subset)

df_imputed = pd.concat(processed_groups).sort_index()

# Pembulatan format klinis standar
df_imputed['UMUR_TAHUN'] = df_imputed['UMUR_TAHUN'].round(0)
numeric_cols = [c for c in df_imputed.columns if c not in ['JENIS_KELAMIN', 'UMUR_TAHUN', 'NM_UNIT']]
df_imputed[numeric_cols] = df_imputed[numeric_cols].round(1)
print('Imputasi selesai tanpa nilai artifisial nol!')

=== Global Population Medians (Fallback) ===
JENIS_KELAMIN         1.00
UMUR_TAHUN           47.00
cholesterol total   190.00
creatinin             1.14
fbs                 125.00
rbs                 140.00
hgb                  12.10
lymfosit%            21.60
mch                  28.30
mchc                 33.00
mcv                  85.50
ureum                39.00
wbc                   9.00
dtype: float64
Imputasi selesai tanpa nilai artifisial nol!


## 7. Verifikasi Ketiadaan Nilai Nol Fisiologis & Simpan `merge_DocLab_imputed.csv`

In [22]:
print('=== Pengecekan Nilai Nol (0) per Parameter ===')
for col in df_imputed.columns:
    zero_count = (df_imputed[col] == 0).sum()
    zero_pct = (zero_count / len(df_imputed)) * 100
    print(f'{col:25s}: {zero_count:5d} ({zero_pct:5.2f}%)')

print('\n=== Ringkasan Statistik Fitur Klinis ===')
df_imputed.describe()

# Simpan dataset akhir yang bersih dan terimputasi
df_imputed.to_csv('merge_DocLab_imputed.csv', index=False)
print('Berhasil menyimpan: merge_DocLab_imputed.csv')

=== Pengecekan Nilai Nol (0) per Parameter ===
JENIS_KELAMIN            :  1649 (47.87%)
UMUR_TAHUN               :     0 ( 0.00%)
cholesterol total        :     0 ( 0.00%)
creatinin                :     0 ( 0.00%)
fbs                      :     0 ( 0.00%)
rbs                      :     0 ( 0.00%)
hgb                      :     0 ( 0.00%)
lymfosit%                :     0 ( 0.00%)
mch                      :     0 ( 0.00%)
mchc                     :     0 ( 0.00%)
mcv                      :     0 ( 0.00%)
ureum                    :     0 ( 0.00%)
wbc                      :     0 ( 0.00%)
NM_UNIT                  :  2812 (81.63%)

=== Ringkasan Statistik Fitur Klinis ===
Berhasil menyimpan: merge_DocLab_imputed.csv
